# Multi-Faceted Filtering — Metadata-Driven Retrieval Refinement

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/multi_faceted_filtering.ipynb)

## Beyond Semantic Similarity

Real-world RAG systems don't just retrieve by semantic similarity.
They *filter* on structured metadata — dates, categories, access
controls, regions — and then combine those hard constraints with
semantic search to surface documents that are both **relevant** and
**appropriate** for the user's context.

A question about *"current Python coding standards"* should not
return a 2019 style guide when a 2024 version exists. A junior
analyst should not receive board-level financial projections, even
if the query is semantically close. Metadata filtering is the
mechanism that enforces these real-world constraints.

This notebook builds a complete **metadata-driven retrieval
pipeline** from scratch, including pre-filtering, post-filtering,
hybrid scoring, and LLM-powered dynamic filter extraction.

### What This Notebook Covers

| Section | Topic |
|---------|-------|
| §1 | Learning Objectives |
| §2 | Why Pure Semantic Search Isn't Enough |
| §3 | Metadata Taxonomy — types, examples, operations |
| §4 | Pre-Filtering vs Post-Filtering strategies |
| §5 | Implementation — document schema, index, retrieval functions |
| §6 | Hybrid Filtering with metadata-aware scoring |
| §7 | LLM-Powered Dynamic Filter Extraction |
| §8 | End-to-End RAG Pipeline with filtering |
| §9 | Evaluation — precision/recall impact of filtering |
| §10 | Synthesis — exercises, takeaways, references |

### Components

| Component | Choice |
|-----------|--------|
| **LLM** | Qwen/Qwen3-8B (4-bit quantised) |
| **Embeddings** | BAAI/bge-base-en-v1.5 |
| **Vector Store** | FAISS (IndexFlatIP) |
| **Filtering** | Pre-filter, Post-filter, Hybrid |
| **Data** | Synthetic enterprise knowledge base (25+ docs) |

> **Runtime note:** This notebook is designed for Google Colab with
> a T4 GPU. The 4-bit quantised Qwen3-8B model requires ~6 GB of
> VRAM. CPU-only execution is possible but will be significantly
> slower for generation steps.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **why pure semantic search fails** in production RAG systems
- Build **metadata-enriched document collections** with typed attributes
- Implement **pre-filtering** and **post-filtering** retrieval strategies
- Combine metadata filters with **semantic relevance scores** (hybrid approach)
- Use an LLM to **dynamically generate filter criteria** from natural language queries
- Evaluate the **precision/recall impact** of metadata filtering on retrieval quality

---

# §2 — Why Pure Semantic Search Isn't Enough

## 2.1 — The Similarity ≠ Relevance Problem

Semantic search finds documents that are *conceptually close* to a
query in embedding space. This is powerful — it handles synonyms,
paraphrases, and even cross-lingual matching. But **similarity is
not the same as relevance in context**. A document can be highly
similar to a query and still be completely wrong for the user.

Consider these failure modes:

**Temporal staleness.** A 2019 remote-work policy is semantically
almost identical to a 2024 query about *"current remote work
guidelines."* Pure semantic search cannot distinguish between them
because the *concepts* are the same — only the *dates* differ.
The 2019 document may contain outdated rules that were superseded
years ago, yet it ranks just as highly as the current version.

**Wrong scope.** A company's HR policy for the US office is
irrelevant to an employee in the UK office asking about parental
leave. Both documents discuss *"parental leave policy,"* but
the statutory requirements, benefit levels, and processes are
completely different. Without a *region* filter, the retriever
cannot distinguish between them.

**Access-control violations.** An intern searching for *"Q3
revenue forecast"* should not retrieve the board-level strategic
projections document, even though it is the most semantically
relevant result. Security levels encode *who may see what*, and
embedding similarity has no concept of authorization.

**Category confusion.** A query about *"Python performance
optimization"* could retrieve a document about web-server tuning
when the user actually wants ML training-loop optimization. Both
involve Python and performance, but they belong to different
departments and document categories.

## 2.2 — A Real-World Example

Imagine an enterprise knowledge base where the phrase *"all
employees must complete annual compliance training"* appears in:

- An **HR policy** (2024, global, internal) — the current rule
- A **legal memo** (2022, US-only, confidential) — discussing
  regulatory obligations
- An **operations report** (2020, EU, public) — reviewing
  historical completion rates

Pure semantic search returns all three with similar scores. But
if the user is a UK employee asking *"Do I need to do compliance
training this year?"*, only the first document is relevant.
Metadata filtering — on **date**, **region**, and **document
type** — is the mechanism that resolves this ambiguity.

> **The fix:** Combine semantic similarity with structured
> metadata filtering. Let embeddings handle *what the document
> is about*, and let metadata handle *whether this particular
> document is appropriate for this particular user and query*.

---

# §3 — Metadata Taxonomy

## 3.1 — Types of Metadata

Not all metadata is created equal. Different types support
different filter operations and serve different purposes in a
retrieval pipeline. The table below categorises the most common
metadata types you will encounter in enterprise RAG systems.

| Metadata Type | Examples | Filter Operations |
|---------------|----------|-------------------|
| **Temporal** | `created_date`, `updated_date`, `valid_until` | Range (before / after / between) |
| **Categorical** | `department`, `document_type`, `language` | Exact match, set membership |
| **Hierarchical** | `org_unit` (Company → Division → Team) | Prefix match, ancestor / descendant |
| **Access Control** | `security_level`, `owner`, `allowed_roles` | Level comparison, set intersection |
| **Source** | `system_of_origin`, `author`, `version` | Exact match |
| **Tagging** | `tags`, `topics`, `keywords` | Set intersection, subset match |

## 3.2 — When Each Type Matters

**Temporal metadata** is essential whenever documents are versioned
or expire. Policies, regulations, and market reports all have a
shelf life. A `date_after` filter ensures the retriever only
considers documents created after a threshold, while `valid_until`
can automatically exclude expired content.

**Categorical metadata** is the most commonly used filter type.
Restricting results to a specific `department` or `document_type`
dramatically reduces noise. When a user asks for *"engineering
policies,"* there is no reason to search across HR memos.

**Hierarchical metadata** encodes organisational structure. A
query scoped to the *"Backend"* team should also match documents
tagged at the *"Engineering"* division level (its ancestor), but
not documents tagged to *"Frontend"* (a sibling).

**Access-control metadata** enforces authorization. This is a
*hard filter* — unlike department or date, it should never be
relaxed. If a user's clearance is level 2, documents at level 3
or 4 must be excluded regardless of semantic relevance.

**Source metadata** tracks provenance. Filtering by `author` or
`system_of_origin` is useful when users trust certain sources
more than others, or when auditing retrieval decisions.

**Tagging metadata** provides flexible, multi-label
classification. A document about *"Python microservices"* might
be tagged `["python", "microservices", "api"]`, enabling
fine-grained filtering that categorical fields alone cannot
express.

---

# §4 — Pre-Filtering vs Post-Filtering

## 4.1 — Two Strategies, Different Trade-offs

There are two fundamental strategies for combining metadata
filters with semantic search, plus a hybrid approach that blends
both.

### Pre-Filtering (Filter → Retrieve)

Apply metadata filters **first**, reducing the document set to
only those that satisfy all constraints. Then perform semantic
search within this filtered subset.

- ✅ **Fast** — semantic search runs on a smaller collection
- ✅ **Perfect metadata precision** — every result satisfies the filters
- ❌ **May miss relevant docs** that barely fail a filter criterion
- ❌ **Requires careful index design** or full-scan filtering

### Post-Filtering (Retrieve → Filter)

Perform a broad semantic search **first**, retrieving more
candidates than needed. Then apply metadata filters to the
results.

- ✅ **Simple to implement** — just a filter pass on results
- ✅ **Never misses semantically relevant docs** in the initial set
- ❌ **Wastes compute** on documents that will be filtered out
- ❌ **May return fewer results** than `top_k` after filtering

### Hybrid (Retrieve → Re-score)

Retrieve a broad set, then **re-rank** using a combined score
that weights both semantic similarity and metadata relevance.
Hard constraints (e.g., security level) are still enforced as
absolute filters, but soft preferences (e.g., recency, preferred
department) become scoring bonuses rather than binary gates.

- ✅ **Best of both worlds** — no hard cutoffs on soft criteria
- ✅ **Tunable** — adjust weights per use case
- ❌ **More complex** to implement and tune

## 4.2 — Strategy Diagrams

```
Pre-filter:  [All Docs] ──filter by metadata──▸ [Subset] ──semantic search──▸ [Results]

Post-filter: [All Docs] ──semantic search──▸ [Candidates] ──filter by metadata──▸ [Results]

Hybrid:      [All Docs] ──semantic search──▸ [Candidates] ──metadata re-scoring──▸ [Results]
```

> **When to use which?** Pre-filtering is best when metadata
> constraints are strict and well-defined (e.g., access control).
> Post-filtering works well when filters are optional or when
> you want to guarantee a minimum number of semantic matches.
> Hybrid is ideal when some metadata is a *preference* rather
> than a *requirement*.

In [ ]:
!pip install -q sentence-transformers faiss-cpu datasets \
    transformers torch bitsandbytes accelerate

In [ ]:
import torch
import json
import numpy as np
import faiss
from datetime import datetime, timedelta
from typing import Any
from dataclasses import dataclass, field, asdict
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

print("✓ All imports successful")

---

# §5 — Implementation

## 5.1 — Document Schema

We define a `Document` dataclass that pairs free-text content
with structured metadata. Each metadata field corresponds to one
of the taxonomy types from §3. The `security_level` field uses
integer encoding (1 = public, 2 = internal, 3 = confidential,
4 = restricted) so that access-control checks reduce to a simple
numeric comparison.

In [ ]:
@dataclass
class Document:
    """A document with rich metadata for multi-faceted filtering."""
    doc_id: str
    content: str
    department: str           # "engineering", "hr", "finance", "legal"
    document_type: str        # "policy", "tutorial", "report", "memo"
    created_date: str         # ISO format YYYY-MM-DD
    security_level: int       # 1=public, 2=internal, 3=confidential, 4=restricted
    author: str
    tags: list[str] = field(default_factory=list)
    region: str = "global"    # "us", "eu", "apac", "global"

    def matches_security(self, user_level: int) -> bool:
        return self.security_level <= user_level

    def __repr__(self) -> str:
        return (
            f"Doc({self.doc_id} | {self.department}/{self.document_type} "
            f"| {self.created_date} | L{self.security_level} | {self.region})"
        )

## 5.2 — Synthetic Enterprise Knowledge Base

We create a realistic collection of 25+ documents spanning
multiple departments, date ranges (2019–2024), security levels,
and regions. This diversity is essential for demonstrating how
metadata filtering changes retrieval behaviour.

In [ ]:
documents = [
    # ── Engineering ─────────────────────────────────────────
    Document(
        doc_id="eng-001",
        content=(
            "Our Python microservices should use FastAPI for all new "
            "REST endpoints. Flask is permitted for internal tools only. "
            "All services must include health check endpoints at "
            "/health and /ready."
        ),
        department="engineering", document_type="policy",
        created_date="2024-06-15", security_level=2, author="Sarah Chen",
        tags=["python", "microservices", "api"], region="global",
    ),
    Document(
        doc_id="eng-002",
        content=(
            "GPU training jobs must use mixed-precision (fp16) by default. "
            "Single-precision (fp32) is only permitted when numerical "
            "stability requires it. All training scripts must log to "
            "Weights & Biases."
        ),
        department="engineering", document_type="policy",
        created_date="2024-03-01", security_level=2, author="James Liu",
        tags=["ml", "training", "gpu", "python"], region="global",
    ),
    Document(
        doc_id="eng-003",
        content=(
            "This tutorial covers setting up a local development "
            "environment with Docker Compose, including PostgreSQL, "
            "Redis, and the main application container with hot-reload."
        ),
        department="engineering", document_type="tutorial",
        created_date="2024-01-20", security_level=1, author="Maria Lopez",
        tags=["docker", "devops", "setup"], region="global",
    ),
    Document(
        doc_id="eng-004",
        content=(
            "Python code must follow PEP 8 with a maximum line length "
            "of 100 characters. Type hints are required for all public "
            "functions. Use ruff for linting and black for formatting."
        ),
        department="engineering", document_type="policy",
        created_date="2019-08-10", security_level=1, author="Tom Baker",
        tags=["python", "style", "linting"], region="global",
    ),
    Document(
        doc_id="eng-005",
        content=(
            "All Python code must use type hints and pass mypy strict "
            "mode. New projects should target Python 3.12+. Dataclasses "
            "are preferred over plain dicts for structured data."
        ),
        department="engineering", document_type="policy",
        created_date="2024-07-01", security_level=2, author="Sarah Chen",
        tags=["python", "typing", "standards"], region="global",
    ),
    Document(
        doc_id="eng-006",
        content=(
            "The Q3 2024 platform reliability report shows 99.95% "
            "uptime across all production services. Three P1 incidents "
            "occurred, all resolved within the 15-minute SLA."
        ),
        department="engineering", document_type="report",
        created_date="2024-10-05", security_level=3, author="James Liu",
        tags=["reliability", "sre", "incidents"], region="global",
    ),
    Document(
        doc_id="eng-007",
        content=(
            "Frontend applications must use React 18+ with TypeScript. "
            "State management should use Zustand for new projects. "
            "Redux is maintained but not recommended for new work."
        ),
        department="engineering", document_type="policy",
        created_date="2024-08-01", security_level=2, author="Maria Lopez",
        tags=["frontend", "react", "typescript"], region="global",
    ),
    # ── HR ──────────────────────────────────────────────────
    Document(
        doc_id="hr-001",
        content=(
            "Employees may work remotely up to 3 days per week. "
            "Core collaboration hours are 10am-3pm local time. Managers "
            "must approve fully-remote arrangements exceeding 2 weeks."
        ),
        department="hr", document_type="policy",
        created_date="2024-02-01", security_level=1, author="Lisa Park",
        tags=["remote", "flexibility", "work-life"], region="global",
    ),
    Document(
        doc_id="hr-002",
        content=(
            "Remote work is not permitted. All employees must be "
            "present in the office five days per week. Exceptions "
            "require VP-level approval with documented justification."
        ),
        department="hr", document_type="policy",
        created_date="2019-06-15", security_level=1, author="John Smith",
        tags=["remote", "office", "attendance"], region="global",
    ),
    Document(
        doc_id="hr-003",
        content=(
            "US employees are entitled to 20 days of paid time off "
            "per year, accrued monthly. Unused PTO may be carried "
            "over up to 5 days into the following calendar year."
        ),
        department="hr", document_type="policy",
        created_date="2024-01-01", security_level=1, author="Lisa Park",
        tags=["pto", "leave", "benefits"], region="us",
    ),
    Document(
        doc_id="hr-004",
        content=(
            "EU employees are entitled to a minimum of 25 days paid "
            "annual leave, in accordance with the EU Working Time "
            "Directive. Public holidays are additional and vary by country."
        ),
        department="hr", document_type="policy",
        created_date="2024-01-01", security_level=1, author="Anna Mueller",
        tags=["pto", "leave", "benefits"], region="eu",
    ),
    Document(
        doc_id="hr-005",
        content=(
            "Annual compensation review results are confidential. "
            "Managers will receive individual adjustment letters by "
            "March 15. Discussing specific figures outside 1:1s is discouraged."
        ),
        department="hr", document_type="memo",
        created_date="2024-02-20", security_level=3, author="Lisa Park",
        tags=["compensation", "review", "salary"], region="global",
    ),
    Document(
        doc_id="hr-006",
        content=(
            "All employees must complete annual compliance training "
            "by December 31. The 2024 curriculum includes data privacy, "
            "anti-harassment, and information security modules."
        ),
        department="hr", document_type="policy",
        created_date="2024-09-01", security_level=1, author="Lisa Park",
        tags=["compliance", "training", "mandatory"], region="global",
    ),
    Document(
        doc_id="hr-007",
        content=(
            "APAC parental leave policy: Primary caregivers receive "
            "16 weeks paid leave. Secondary caregivers receive 4 weeks. "
            "Leave may be taken flexibly within 12 months of birth."
        ),
        department="hr", document_type="policy",
        created_date="2024-01-15", security_level=1, author="Yuki Tanaka",
        tags=["parental", "leave", "benefits"], region="apac",
    ),
    # ── Finance ─────────────────────────────────────────────
    Document(
        doc_id="fin-001",
        content=(
            "Q1 2024 revenue was $42.3M, up 18% YoY. Operating "
            "margin improved to 23.1% driven by infrastructure cost "
            "optimisation and headcount efficiency."
        ),
        department="finance", document_type="report",
        created_date="2024-04-15", security_level=4, author="CFO Office",
        tags=["revenue", "quarterly", "earnings"], region="global",
    ),
    Document(
        doc_id="fin-002",
        content=(
            "Expense reports must be submitted within 30 days of "
            "the transaction. Receipts are required for all expenses "
            "over $25. International travel requires pre-approval."
        ),
        department="finance", document_type="policy",
        created_date="2023-07-01", security_level=1, author="Mark Davis",
        tags=["expenses", "travel", "reimbursement"], region="global",
    ),
    Document(
        doc_id="fin-003",
        content=(
            "The 2025 budget allocates $12M to engineering (up 15%), "
            "$3.2M to marketing, and $1.8M to HR. Capital expenditure "
            "is capped at $5M with board approval required above $2M."
        ),
        department="finance", document_type="report",
        created_date="2024-11-01", security_level=4, author="CFO Office",
        tags=["budget", "planning", "2025"], region="global",
    ),
    Document(
        doc_id="fin-004",
        content=(
            "Vendor payments are processed on the 1st and 15th of "
            "each month. Net-30 terms apply to all new vendor contracts. "
            "Invoices must include a valid PO number."
        ),
        department="finance", document_type="policy",
        created_date="2023-03-15", security_level=2, author="Mark Davis",
        tags=["vendors", "payments", "procurement"], region="global",
    ),
    Document(
        doc_id="fin-005",
        content=(
            "Annual financial audit for FY2023 found no material "
            "weaknesses. Two minor observations regarding expense "
            "receipt documentation were noted and addressed."
        ),
        department="finance", document_type="report",
        created_date="2024-03-30", security_level=3, author="CFO Office",
        tags=["audit", "compliance", "annual"], region="global",
    ),
    # ── Legal ───────────────────────────────────────────────
    Document(
        doc_id="leg-001",
        content=(
            "All customer data processing activities must comply "
            "with GDPR. Data processing agreements (DPAs) must be "
            "signed before sharing personal data with third parties."
        ),
        department="legal", document_type="policy",
        created_date="2024-05-01", security_level=2, author="Eva Schmidt",
        tags=["gdpr", "privacy", "compliance", "data"], region="eu",
    ),
    Document(
        doc_id="leg-002",
        content=(
            "The standard NDA has been updated to include AI-generated "
            "content provisions. All new NDAs effective July 2024 must "
            "use the v3.2 template from the legal portal."
        ),
        department="legal", document_type="memo",
        created_date="2024-06-20", security_level=3, author="Eva Schmidt",
        tags=["nda", "contracts", "ai"], region="global",
    ),
    Document(
        doc_id="leg-003",
        content=(
            "CCPA compliance requires that California residents can "
            "request deletion of their personal data within 45 days. "
            "The data-rights portal must be linked from the privacy page."
        ),
        department="legal", document_type="policy",
        created_date="2023-09-01", security_level=2, author="David Kim",
        tags=["ccpa", "privacy", "compliance", "data"], region="us",
    ),
    # ── Marketing ───────────────────────────────────────────
    Document(
        doc_id="mkt-001",
        content=(
            "Brand guidelines require the use of Montserrat for "
            "headings and Inter for body text. The primary brand colour "
            "is #2563EB (blue-600). Logo minimum size is 32px height."
        ),
        department="marketing", document_type="guide",
        created_date="2024-03-10", security_level=1, author="Amy Zhao",
        tags=["brand", "design", "visual-identity"], region="global",
    ),
    Document(
        doc_id="mkt-002",
        content=(
            "The Q2 2024 campaign delivered 2.3M impressions with "
            "a 3.1% CTR. Cost per acquisition dropped to $18.40 "
            "from $23.10 in Q1, driven by improved targeting."
        ),
        department="marketing", document_type="report",
        created_date="2024-07-15", security_level=2, author="Amy Zhao",
        tags=["campaign", "metrics", "performance"], region="us",
    ),
    # ── Operations ──────────────────────────────────────────
    Document(
        doc_id="ops-001",
        content=(
            "All production deployments must go through the CI/CD "
            "pipeline. Manual deployments are prohibited. Rollback "
            "procedures must be tested before each release window."
        ),
        department="operations", document_type="policy",
        created_date="2024-04-01", security_level=2, author="Raj Patel",
        tags=["deployment", "cicd", "release"], region="global",
    ),
    Document(
        doc_id="ops-002",
        content=(
            "APAC datacenter migration is scheduled for Q4 2024. "
            "All services must be dual-region capable by September 30. "
            "Latency targets: <100ms p99 for Singapore, <150ms for Tokyo."
        ),
        department="operations", document_type="memo",
        created_date="2024-08-15", security_level=3, author="Raj Patel",
        tags=["infrastructure", "migration", "apac"], region="apac",
    ),
    Document(
        doc_id="ops-003",
        content=(
            "Incident response procedure: P1 incidents require "
            "immediate page to on-call engineer. War room opens within "
            "5 minutes. Status page updated within 10 minutes."
        ),
        department="operations", document_type="guide",
        created_date="2024-05-20", security_level=2, author="Raj Patel",
        tags=["incidents", "oncall", "sre"], region="global",
    ),
]

print(f"✓ Created {len(documents)} documents")
print(f"  Departments: {sorted(set(d.department for d in documents))}")
print(f"  Types: {sorted(set(d.document_type for d in documents))}")
print(f"  Date range: {min(d.created_date for d in documents)}"
      f" → {max(d.created_date for d in documents)}")
print(f"  Regions: {sorted(set(d.region for d in documents))}")

## 5.3 — Building the Embedding Index

We use `BAAI/bge-base-en-v1.5` to embed each document's content
and build a FAISS inner-product index. Since the embeddings are
L2-normalised, inner product equals cosine similarity.

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

contents = [doc.content for doc in documents]
embeddings = embed_model.encode(
    contents, normalize_embeddings=True, show_progress_bar=True
)
embeddings = np.array(embeddings, dtype=np.float32)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"✓ FAISS index built: {index.ntotal} vectors, dim={dim}")

## 5.4 — Pre-Filter Strategy

The pre-filter approach applies all metadata constraints **before**
any semantic computation. We iterate over the full document list,
retain only those passing every filter, then compute semantic
scores exclusively within this filtered subset.

In [ ]:
def pre_filter_retrieve(
    query: str,
    top_k: int = 5,
    department: str | None = None,
    document_type: str | None = None,
    date_after: str | None = None,
    date_before: str | None = None,
    max_security_level: int = 4,
    region: str | None = None,
    required_tags: list[str] | None = None,
) -> list[tuple[Document, float]]:
    """Pre-filter documents by metadata, then rank by semantic similarity."""
    # Step 1: Apply metadata filters
    candidates = []
    for i, doc in enumerate(documents):
        if department and doc.department != department:
            continue
        if document_type and doc.document_type != document_type:
            continue
        if date_after and doc.created_date < date_after:
            continue
        if date_before and doc.created_date > date_before:
            continue
        if doc.security_level > max_security_level:
            continue
        if region and doc.region != region and doc.region != "global":
            continue
        if required_tags and not set(required_tags).issubset(set(doc.tags)):
            continue
        candidates.append((i, doc))

    if not candidates:
        print("⚠ No documents matched the metadata filters.")
        return []

    # Step 2: Semantic search within filtered subset
    query_emb = embed_model.encode([query], normalize_embeddings=True)
    candidate_indices = [idx for idx, _ in candidates]
    candidate_embs = embeddings[candidate_indices]
    scores = (candidate_embs @ query_emb.T).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(candidates[i][1], float(scores[i])) for i in top_indices]


# Quick test
results = pre_filter_retrieve(
    "Python coding standards",
    department="engineering",
    date_after="2023-01-01",
)
for doc, score in results:
    print(f"  {score:.4f}  {doc}")

## 5.5 — Post-Filter Strategy

The post-filter approach retrieves a broader set of candidates
using semantic search first (`initial_k` > `final_k`), then
applies metadata filters to prune irrelevant results. This
guarantees we do not miss semantically strong matches, but may
return fewer than `final_k` results if many candidates fail
the filters.

In [ ]:
def post_filter_retrieve(
    query: str,
    initial_k: int = 15,
    final_k: int = 5,
    department: str | None = None,
    document_type: str | None = None,
    date_after: str | None = None,
    date_before: str | None = None,
    max_security_level: int = 4,
    region: str | None = None,
) -> list[tuple[Document, float]]:
    """Retrieve by semantic similarity first, then filter by metadata."""
    # Step 1: Broad semantic search
    query_emb = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(
        np.array(query_emb, dtype=np.float32), initial_k
    )

    # Step 2: Apply metadata filters to results
    filtered = []
    for score, idx in zip(scores[0], indices[0]):
        doc = documents[idx]
        if department and doc.department != department:
            continue
        if document_type and doc.document_type != document_type:
            continue
        if date_after and doc.created_date < date_after:
            continue
        if date_before and doc.created_date > date_before:
            continue
        if doc.security_level > max_security_level:
            continue
        if region and doc.region != region and doc.region != "global":
            continue
        filtered.append((doc, float(score)))
        if len(filtered) >= final_k:
            break

    return filtered


# Quick test
results = post_filter_retrieve(
    "Python coding standards",
    department="engineering",
    date_after="2023-01-01",
)
for doc, score in results:
    print(f"  {score:.4f}  {doc}")

## 5.6 — Side-by-Side Comparison

Let's run the same query and filters through both strategies to
see how they differ in practice. The key observation is that
pre-filtering searches a smaller space (faster, guaranteed filter
satisfaction) while post-filtering preserves the original semantic
ranking but may lose results to filtering.

In [ ]:
test_query = "What are the Python coding standards?"
filters = {
    "department": "engineering",
    "date_after": "2023-01-01",
}

pre_results = pre_filter_retrieve(test_query, **filters)
post_results = post_filter_retrieve(test_query, **filters)

print("=" * 70)
print("PRE-FILTER RESULTS")
print("=" * 70)
for doc, score in pre_results:
    print(f"  {score:.4f}  {doc}")

print(f"\n{'=' * 70}")
print("POST-FILTER RESULTS")
print("=" * 70)
for doc, score in post_results:
    print(f"  {score:.4f}  {doc}")

# Check overlap
pre_ids = {doc.doc_id for doc, _ in pre_results}
post_ids = {doc.doc_id for doc, _ in post_results}
print(f"\nOverlap: {pre_ids & post_ids}")
print(f"Only in pre-filter:  {pre_ids - post_ids}")
print(f"Only in post-filter: {post_ids - pre_ids}")

---

# §6 — Hybrid Filtering with Metadata-Aware Scoring

## 6.1 — Soft Preferences vs Hard Constraints

The pre- and post-filter strategies treat every metadata
criterion as a binary gate: a document either passes or it does
not. But in practice, some metadata is a *hard constraint*
(security level — non-negotiable) while other metadata is a
*soft preference* (department — prefer engineering, but an
operations doc about deployment pipelines could also be useful).

The hybrid approach keeps hard filters as absolute exclusions
but converts soft criteria into **scoring bonuses**. A document
from the preferred department gets a metadata boost; a recent
document gets a recency bonus. The final score is a weighted
combination of semantic similarity and metadata relevance:

$$\text{score} = (1 - w) \cdot \text{sim}_{\text{semantic}} + w \cdot \text{score}_{\text{metadata}}$$

where $w$ is the `metadata_weight` hyperparameter (default 0.2).

In [ ]:
def hybrid_retrieve(
    query: str,
    top_k: int = 5,
    metadata_weight: float = 0.2,
    preferred_department: str | None = None,
    preferred_date_after: str | None = None,
    max_security_level: int = 4,
    preferred_region: str | None = None,
) -> list[tuple[Document, float]]:
    """Combine semantic similarity with metadata relevance scoring."""
    query_emb = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(
        np.array(query_emb, dtype=np.float32), 20
    )

    results = []
    for sem_score, idx in zip(scores[0], indices[0]):
        doc = documents[idx]

        # Hard filter: security level is non-negotiable
        if doc.security_level > max_security_level:
            continue

        # Soft metadata scoring
        meta_score = 0.0

        # Department match bonus
        if preferred_department and doc.department == preferred_department:
            meta_score += 0.5

        # Region match bonus
        if preferred_region:
            if doc.region == preferred_region or doc.region == "global":
                meta_score += 0.2

        # Recency bonus with 5-year decay
        if preferred_date_after:
            if doc.created_date >= preferred_date_after:
                meta_score += 0.3
        days_old = (
            datetime.now() - datetime.fromisoformat(doc.created_date)
        ).days
        meta_score += max(0, 0.2 * (1 - days_old / 1825))

        combined = (
            (1 - metadata_weight) * float(sem_score)
            + metadata_weight * meta_score
        )
        results.append((doc, combined))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

## 6.2 — Hybrid vs Pure Semantic Search

A date-sensitive query like *"What is the current remote work
policy?"* is the perfect test case. Pure semantic search may
rank the **2019 no-remote-work policy** (`hr-002`) alongside the
**2024 flexible policy** (`hr-001`) because both discuss remote
work. The hybrid scorer should push the 2024 version to the top
thanks to its recency bonus.

In [ ]:
# Pure semantic search (no metadata scoring)
query = "What is the current remote work policy?"
query_emb = embed_model.encode([query], normalize_embeddings=True)
scores, indices = index.search(
    np.array(query_emb, dtype=np.float32), 5
)

print("=" * 70)
print("PURE SEMANTIC SEARCH")
print("=" * 70)
for score, idx in zip(scores[0], indices[0]):
    print(f"  {score:.4f}  {documents[idx]}")

# Hybrid search with recency preference
hybrid_results = hybrid_retrieve(
    query,
    preferred_department="hr",
    preferred_date_after="2023-01-01",
    max_security_level=2,
)

print(f"\n{'=' * 70}")
print("HYBRID SEARCH (recency + department preference)")
print("=" * 70)
for doc, score in hybrid_results:
    print(f"  {score:.4f}  {doc}")

---

# §7 — LLM-Powered Dynamic Filter Extraction

## 7.1 — From Natural Language to Structured Filters

So far we have specified filters manually. In a production system,
users type natural language queries like *"Show me engineering
policies from this year"* and expect the system to figure out
that `department=engineering`, `document_type=policy`, and
`date_after=2024-01-01`.

We use the LLM to **parse the query into a structured filter
dictionary**, which is then fed into our retrieval functions.
This is a lightweight form of *intent recognition* that turns
unstructured input into structured API parameters.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"✓ Model loaded: {MODEL_ID} (4-bit)")

## 7.2 — Filter Extraction Prompt

The prompt instructs the model to output a JSON object with only
the filters that are **implied** by the query. If the query does
not mention a time range, the `date_after` / `date_before` keys
should be omitted entirely. This avoids over-constraining the
retrieval with hallucinated filters.

In [ ]:
FILTER_EXTRACTION_PROMPT = """You are a query analyzer for a
document retrieval system. Extract structured metadata filters
from the user's natural language query.

Available filters:
- department: engineering, hr, finance, legal, marketing, operations
- document_type: policy, tutorial, report, memo, guide
- date_after: YYYY-MM-DD (documents created after this date)
- date_before: YYYY-MM-DD (documents created before this date)
- max_security_level: 1-4 (1=public, 2=internal, 3=confidential, 4=restricted)
- region: us, eu, apac, global
- required_tags: list of relevant tags

Rules:
- Only include filters that are clearly implied by the query.
- If the query says "this year" or "current", set date_after to 2024-01-01.
- If no filter is implied, return an empty JSON object {{}}.
- Respond ONLY with a valid JSON object. No explanation.

Query: {query}
Filters (JSON):"""


def extract_filters(query: str) -> dict:
    """Use the LLM to extract structured filters from a natural language query."""
    prompt = FILTER_EXTRACTION_PROMPT.format(query=query)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
        )

    generated = tokenizer.decode(
        output_ids[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Extract JSON from the response
    try:
        return json.loads(generated)
    except json.JSONDecodeError:
        import re
        match = re.search(r"\{[^{}]*\}", generated, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        print(f"⚠ Could not parse filters from: {generated[:200]}")
        return {}

## 7.3 — Testing Filter Extraction

We test the filter extractor on a diverse set of queries to see
if the LLM correctly infers the appropriate metadata constraints.

In [ ]:
test_queries = [
    "Show me engineering policies from this year",
    "What are the HR guidelines for EU employees?",
    "Find confidential financial reports from Q1 2024",
    "What public tutorials do we have about Python?",
    "Tell me about the APAC datacenter migration",
]

for q in test_queries:
    filters = extract_filters(q)
    print(f"Query:     {q}")
    print(f"Extracted: {json.dumps(filters, indent=2)}")
    print()

---

# §8 — End-to-End RAG Pipeline with Filtering

## 8.1 — Putting It All Together

The complete pipeline has three stages:

1. **Extract** structured filters from the natural language query
2. **Retrieve** documents using hybrid filtering (semantic +
   metadata scoring)
3. **Generate** an answer grounded in the retrieved context

The `user_security_level` parameter simulates access control:
the pipeline clamps the `max_security_level` filter to the
user's clearance, ensuring no over-privileged retrieval.

In [ ]:
def generate_answer(query: str, contexts: list[str]) -> str:
    """Generate a grounded answer using the LLM and retrieved contexts."""
    context_block = "\n".join(
        f"[{i+1}] {c}" for i, c in enumerate(contexts)
    )
    prompt = (
        "Answer the question based ONLY on the provided context. "
        "If the context does not contain enough information, say so.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\nAnswer:"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
        )

    return tokenizer.decode(
        output_ids[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

In [ ]:
def full_rag_pipeline(
    query: str, user_security_level: int = 2
) -> tuple[str, list[tuple[Document, float]], dict]:
    """Complete pipeline: extract filters -> retrieve -> generate answer."""
    # Step 1: Extract filters from natural language query
    filters = extract_filters(query)
    print(f"Extracted filters: {json.dumps(filters, indent=2)}")

    # Step 2: Enforce access control ceiling
    filters["max_security_level"] = min(
        filters.get("max_security_level", 4), user_security_level
    )

    # Step 3: Map extracted filters to hybrid_retrieve params
    retrieve_params = {
        "preferred_department": filters.get("department"),
        "preferred_date_after": filters.get("date_after"),
        "max_security_level": filters["max_security_level"],
        "preferred_region": filters.get("region"),
    }
    results = hybrid_retrieve(query, **retrieve_params)

    # Step 4: Generate answer from retrieved contexts
    contexts = [doc.content for doc, _ in results]
    answer = generate_answer(query, contexts)

    return answer, results, filters

## 8.2 — Running the Full Pipeline

We run two queries to demonstrate the pipeline end-to-end.
Notice how the extracted filters automatically scope the
retrieval to the right department, date range, and region.

In [ ]:
pipeline_queries = [
    "What is the current remote work policy?",
    "What are the Python coding standards for engineering?",
]

for q in pipeline_queries:
    print("=" * 70)
    print(f"QUERY: {q}")
    print("=" * 70)
    answer, results, filters = full_rag_pipeline(q)

    print(f"\nRetrieved {len(results)} documents:")
    for doc, score in results:
        print(f"  {score:.4f}  {doc}")

    print(f"\nGenerated Answer:\n{answer}")
    print()

---

# §9 — Evaluation: Impact of Filtering on Retrieval Quality

## 9.1 — Precision and Recall

To quantify the impact of metadata filtering, we measure
**precision@k** (what fraction of retrieved documents are
relevant?) and **recall@k** (what fraction of all relevant
documents are retrieved?) with and without filtering.

We define test cases with known ground-truth relevant document
IDs, then compare unfiltered semantic search against filtered
retrieval.

In [ ]:
def precision_at_k(
    retrieved_ids: list[str], relevant_ids: set[str], k: int
) -> float:
    """Fraction of top-k retrieved documents that are relevant."""
    return len(set(retrieved_ids[:k]) & relevant_ids) / k


def recall_at_k(
    retrieved_ids: list[str], relevant_ids: set[str], k: int
) -> float:
    """Fraction of all relevant documents found in top-k."""
    if not relevant_ids:
        return 0.0
    return len(set(retrieved_ids[:k]) & relevant_ids) / len(relevant_ids)


def unfiltered_retrieve(
    query: str, top_k: int = 5
) -> list[tuple[Document, float]]:
    """Pure semantic search with no metadata filtering."""
    query_emb = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(
        np.array(query_emb, dtype=np.float32), top_k
    )
    return [
        (documents[idx], float(score))
        for score, idx in zip(scores[0], indices[0])
    ]

## 9.2 — Test Cases and Results

Each test case defines a query, the set of truly relevant
document IDs, and the appropriate metadata filters. We compare
unfiltered retrieval (pure semantic) against pre-filtered
retrieval to measure the precision and recall impact.

In [ ]:
test_cases = [
    {
        "query": "current Python coding standards",
        "relevant_ids": {"eng-001", "eng-005"},
        "filters": {"department": "engineering", "date_after": "2023-01-01"},
    },
    {
        "query": "remote work policy",
        "relevant_ids": {"hr-001"},
        "filters": {"department": "hr", "date_after": "2023-01-01"},
    },
    {
        "query": "EU employee leave entitlement",
        "relevant_ids": {"hr-004"},
        "filters": {"department": "hr", "region": "eu"},
    },
    {
        "query": "data privacy compliance requirements",
        "relevant_ids": {"leg-001", "leg-003"},
        "filters": {"department": "legal"},
    },
    {
        "query": "production deployment process",
        "relevant_ids": {"ops-001"},
        "filters": {"department": "operations", "document_type": "policy"},
    },
]

K = 5
print(f"{'Query':<45} {'Method':<12} {'P@5':<8} {'R@5':<8}")
print("-" * 75)

for tc in test_cases:
    query = tc["query"]
    relevant = tc["relevant_ids"]
    filters = tc["filters"]

    # Unfiltered
    unf_results = unfiltered_retrieve(query, top_k=K)
    unf_ids = [doc.doc_id for doc, _ in unf_results]
    unf_p = precision_at_k(unf_ids, relevant, K)
    unf_r = recall_at_k(unf_ids, relevant, K)

    # Pre-filtered
    filt_results = pre_filter_retrieve(query, top_k=K, **filters)
    filt_ids = [doc.doc_id for doc, _ in filt_results]
    filt_p = precision_at_k(filt_ids, relevant, K) if filt_ids else 0.0
    filt_r = recall_at_k(filt_ids, relevant, K) if filt_ids else 0.0

    print(f"{query:<45} {'unfiltered':<12} {unf_p:<8.2f} {unf_r:<8.2f}")
    print(f"{'':<45} {'filtered':<12} {filt_p:<8.2f} {filt_r:<8.2f}")

## 9.3 — Discussion

The evaluation results typically show several patterns:

1. **Filtering consistently improves precision.** By excluding
   documents from irrelevant departments or date ranges, every
   retrieved result is more likely to be genuinely useful.

2. **Strict filtering can reduce recall.** If a relevant document
   exists in a different department (e.g., an operations doc that
   answers an engineering question), hard department filters will
   exclude it.

3. **Temporal filtering is high-value.** Date filters eliminate
   the *temporal staleness* problem almost entirely. The 2019
   remote-work policy never appears when `date_after=2023-01-01`.

4. **The hybrid approach balances the trade-off.** By using soft
   preferences instead of hard gates for non-critical metadata,
   hybrid retrieval achieves good precision without sacrificing
   recall.

5. **LLM filter extraction is imperfect.** The model sometimes
   misinterprets ambiguous queries or adds unnecessary filters.
   In production, log and audit the extracted filters, and
   consider letting users review or edit them before retrieval.

> **Production tip:** Always compare filtered retrieval against
> an unfiltered baseline during development. If filtering
> consistently reduces recall below an acceptable threshold,
> relax the hard filters into soft preferences.

---

# §10 — Synthesis

## 10.1 — Exercises

**Exercise 1: Hierarchical Filtering**

Extend the `Document` schema with an `org_path` field (e.g.,
`"company/engineering/backend"`) and implement hierarchical
filtering that matches any level of the hierarchy. A filter for
`"company/engineering"` should match documents tagged at
`"company/engineering/backend"` and
`"company/engineering/frontend"` but not
`"company/marketing"`.

**Exercise 2: Time-Decay Scoring**

Implement a retrieval function where document relevance decays
exponentially over time:

$$\text{decay}(t) = e^{-\lambda t}$$

where $t$ is the document age in days and $\lambda$ is a
configurable decay rate. Compare exponential decay against the
linear decay used in the hybrid scorer.

**Exercise 3: Access-Control Simulation**

Create a role-based access system with the following roles:

| Role | Max Security Level | Departments Visible |
|------|-------------------|---------------------|
| intern | 1 | engineering |
| employee | 2 | all |
| manager | 3 | all |
| executive | 4 | all |

Run the same query as each role and compare the retrieved
documents. Verify that no role ever sees documents above its
clearance level.

**Exercise 4: Filter Validation**

The LLM sometimes generates invalid filter values (e.g.,
`department="science"` which does not exist). Implement a
validation layer that checks extracted filters against the
known schema and either corrects or removes invalid entries.

In [ ]:
# ── Exercise 1 Starter Code ──────────────────────────────────

def hierarchical_filter(
    doc_org_path: str, filter_prefix: str
) -> bool:
    """Return True if doc_org_path falls under filter_prefix."""
    # YOUR CODE HERE
    pass


# ── Exercise 2 Starter Code ──────────────────────────────────
import math

def exponential_decay_score(
    created_date: str, decay_lambda: float = 0.002
) -> float:
    """Compute exponential time-decay score for a document."""
    # YOUR CODE HERE
    pass


# ── Exercise 3 Starter Code ──────────────────────────────────

ROLES = {
    "intern":    {"max_security_level": 1, "departments": {"engineering"}},
    "employee":  {"max_security_level": 2, "departments": None},  # all
    "manager":   {"max_security_level": 3, "departments": None},
    "executive": {"max_security_level": 4, "departments": None},
}

# YOUR CODE HERE: run the same query for each role

## 10.2 — Key Takeaways

1. **Semantic similarity alone is insufficient** for production
   RAG. Metadata filtering handles temporal, categorical,
   hierarchical, and access-control constraints that embeddings
   cannot encode.

2. **Pre-filtering** is best for strict, well-defined constraints
   (security level, access control). It is fast and guarantees
   metadata compliance.

3. **Post-filtering** is simpler to implement and preserves the
   semantic ranking, but may return fewer results than expected.

4. **Hybrid scoring** is the most flexible approach. It treats
   hard constraints as absolute filters and soft preferences as
   scoring bonuses, balancing precision and recall.

5. **LLM-powered filter extraction** bridges the gap between
   natural language queries and structured metadata filters,
   but requires validation and fallback logic.

6. **Always evaluate with and without filtering.** Measure
   precision and recall to ensure filtering is helping, not
   hurting.

7. **Access control is a hard filter, not a preference.** Never
   relax security-level constraints for the sake of recall.

## 10.3 — References

- [Pinecone — Metadata Filtering](https://docs.pinecone.io/docs/metadata-filtering)
  — How Pinecone implements filtered vector search at scale
- [Weaviate — Filtered Vector Search](https://weaviate.io/developers/weaviate/search/filters)
  — Pre-filtering and post-filtering strategies in Weaviate
- [Qdrant — Filtering](https://qdrant.tech/documentation/concepts/filtering/)
  — Payload-based filtering with boolean conditions
- [LlamaIndex — Metadata Filtering](https://docs.llamaindex.ai/en/stable/)
  — Metadata-aware retrieval in LlamaIndex
- [FAISS Wiki](https://github.com/facebookresearch/faiss/wiki)
  — Index types and search strategies

### Related Castalia Notebooks

- [`fusion_retrieval.ipynb`](fusion_retrieval.ipynb)
  — BM25 + dense retrieval with Reciprocal Rank Fusion
- [`reranking.ipynb`](reranking.ipynb)
  — Cross-encoder reranking for improved precision
- [`hierarchical_indices.ipynb`](hierarchical_indices.ipynb)
  — Multi-level index structures for large collections